# Graph-Based Money Mule Detection — Interactive Notebook

This notebook walks through the complete system step by step, explaining every concept with live outputs.

**Contents:**
1. Data generation and loading
2. Transaction graph construction and analysis
3. Feature engineering (node, graph, temporal)
4. Community detection (finding mule networks)
5. Machine learning models (IsolationForest, RandomForest, GradientBoosting)
6. Mule lifecycle stage detection
7. Visualisations and result interpretation
8. API scoring demo


In [ ]:
import sys, os
# Ensure project root is on path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 6)

print('Setup complete. Project root:', PROJECT_ROOT)

## 1. Data Generation

We generate a synthetic AML dataset that simulates the 3-layer mule network model:
- **Layer 0:** Fraud source accounts (inject dirty money)
- **Layer 1:** Mule collector accounts (receive and hold)
- **Layer 2:** Mule distributor accounts (split and forward)
- **Exit:** Cash-out destinations

In [ ]:
from src.ingestion.data_generator import generate_synthetic_dataset

df = generate_synthetic_dataset(
    n_accounts      = 2000,
    n_transactions  = 20000,
    mule_fraction   = 0.04,
    n_mule_networks = 8,
    save=False,
)

print(f'\nDataset shape: {df.shape}')
df.head()

In [ ]:
# Explore the dataset
print('Transaction type distribution:')
print(df['is_fraud'].value_counts().rename({0: 'Normal', 1: 'Fraud'}))
print(f'\nFraud rate: {df["is_fraud"].mean():.2%}')
print(f'Amount stats:\n{df["transaction_amount"].describe().round(2)}')

In [ ]:
# Plot transaction volume over time
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['week'] = df['timestamp'].dt.isocalendar().week.astype(int)

weekly = df.groupby(['week','is_fraud']).size().unstack(fill_value=0)
weekly.columns = ['Normal','Fraud']

fig, ax = plt.subplots()
weekly['Normal'].plot(ax=ax, color='#85B7EB', label='Normal')
weekly['Fraud'].plot(ax=ax, color='#E24B4A', label='Fraud')
ax.set_title('Weekly transaction volume by type', fontsize=13, fontweight='bold')
ax.set_xlabel('Week'); ax.set_ylabel('Transaction count')
ax.legend(frameon=False)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()

## 2. Graph Construction

We build a **directed weighted graph** where:
- **Nodes** = bank accounts
- **Edges** = money flows (sender → receiver)
- **Edge weight** = total transaction amount

Mule networks appear as **dense subgraphs** within the larger transaction network.

In [ ]:
from src.graph.graph_builder import build_transaction_graph, graph_summary

G_summary, G_full = build_transaction_graph(df)

stats = graph_summary(G_summary)
print('\nGraph Statistics:')
for k, v in stats.items():
    print(f'  {k:<22}: {v}')

In [ ]:
# Visualise a sample of the graph
# Red = confirmed mule, Blue = normal
fraud_nodes  = [n for n,d in G_summary.nodes(data=True) if d.get('is_fraud')]
normal_nodes = [n for n,d in G_summary.nodes(data=True) if not d.get('is_fraud')]

# Sample for visibility
rng = np.random.default_rng(42)
sample = set(fraud_nodes) | set(rng.choice(normal_nodes, 150, replace=False).tolist())
sub = G_summary.subgraph(sample).copy()

pos = nx.spring_layout(sub, seed=42, k=0.9)
colors = ['#E24B4A' if sub.nodes[n].get('is_fraud') else '#85B7EB' for n in sub.nodes()]
sizes  = [120 if sub.nodes[n].get('is_fraud') else 30 for n in sub.nodes()]

fig, ax = plt.subplots(figsize=(12, 8))
nx.draw_networkx_edges(sub, pos, ax=ax, alpha=0.15, width=0.5, edge_color='#B4B2A9')
nx.draw_networkx_nodes(sub, pos, ax=ax, node_color=colors, node_size=sizes, alpha=0.85)
ax.set_title('Transaction Graph (red = mule account)', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()

## 3. Feature Engineering

We extract **38 features** per account across three groups:

| Group | Key features | Mule signal |
|-------|-------------|-------------|
| Node behavioral | `passthrough_ratio`, `burst_score`, `drain_rate` | Mules pass money through without keeping it |
| Graph topology | `pagerank`, `betweenness`, `clustering` | Mules are central hubs in the flow graph |
| Temporal | `burst_ratio`, `weekly_cv`, `sudden_wakeup` | Mules show erratic, bursty patterns |

In [ ]:
from src.features.feature_engineering import build_feature_matrix, get_feature_columns

feature_matrix = build_feature_matrix(df, G_summary)
feat_cols      = get_feature_columns(feature_matrix)

print(f'Feature matrix shape: {feature_matrix.shape}')
print(f'Number of features: {len(feat_cols)}')
feature_matrix.head(3)

In [ ]:
# Compare mule vs normal accounts on key features
key_features = ['burst_ratio', 'passthrough_ratio', 'weekly_cv',
                'pagerank', 'betweenness', 'n_sent']

fm = feature_matrix.set_index('account')
mule_feats   = fm[fm['is_fraud'] == 1][key_features]
normal_feats = fm[fm['is_fraud'] == 0][key_features]

comparison = pd.DataFrame({
    'Normal (mean)': normal_feats.mean(),
    'Mule (mean)':   mule_feats.mean(),
    'Mule / Normal': (mule_feats.mean() / (normal_feats.mean() + 1e-9)).round(2),
})
print('Feature comparison — mule vs normal accounts:')
print(comparison.round(4).to_string())

In [ ]:
# Distribution plots for top discriminating features
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    ax = axes[i]
    ax.hist(normal_feats[feat].clip(upper=normal_feats[feat].quantile(0.99)),
            bins=40, alpha=0.6, color='#85B7EB', density=True, label='Normal')
    ax.hist(mule_feats[feat].clip(upper=mule_feats[feat].quantile(0.99)),
            bins=40, alpha=0.7, color='#E24B4A', density=True, label='Mule')
    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(frameon=False, fontsize=8)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Feature Distributions: Mule vs Normal', fontsize=13, fontweight='bold')
plt.tight_layout()

## 4. Community Detection

Money mule networks form **dense communities** in the transaction graph. We use graph clustering to find these groups, then score each community's suspicion level.

A community is flagged as suspicious when it has:
- High internal transaction density
- High average PageRank or betweenness centrality
- High pass-through ratios
- Elevated mule account density (if labels available)

In [ ]:
from src.community.community_detector import (
    detect_communities, score_communities, get_suspicious_accounts
)

partition = detect_communities(G_summary, method='louvain')
comm_df   = score_communities(partition, feature_matrix, G_summary)

print(f'Top 5 most suspicious communities:')
display_cols = ['community_id','size','mule_rate','suspicion_score',
                'avg_burst_ratio','internal_density','is_suspicious']
print(comm_df[display_cols].head(5).to_string(index=False))

In [ ]:
# Visualise the most suspicious community
top_community  = comm_df.iloc[0]
cluster_nodes  = top_community['accounts'][:40]

neighbourhood = set(cluster_nodes)
for n in cluster_nodes:
    neighbourhood.update(list(G_summary.predecessors(n))[:2])
    neighbourhood.update(list(G_summary.successors(n))[:2])

sub_comm = G_summary.subgraph(neighbourhood).copy()
pos      = nx.spring_layout(sub_comm, seed=42, k=1.1)
cluster_set = set(cluster_nodes)

colors = []
for n in sub_comm.nodes():
    if sub_comm.nodes[n].get('is_fraud'):
        colors.append('#E24B4A')
    elif n in cluster_set:
        colors.append('#EF9F27')
    else:
        colors.append('#85B7EB')

fig, ax = plt.subplots(figsize=(10, 8))
nx.draw_networkx_edges(sub_comm, pos, ax=ax, alpha=0.4, arrows=True, arrowsize=10)
nx.draw_networkx_nodes(sub_comm, pos, ax=ax, node_color=colors, node_size=120, alpha=0.9)
ax.set_title(f'Community {int(top_community["community_id"])} — '
             f'Suspicion score: {top_community["suspicion_score"]:.3f}',
             fontsize=13, fontweight='bold')
ax.axis('off')
from matplotlib.patches import Patch
legend = [Patch(color='#E24B4A', label='Confirmed mule'),
          Patch(color='#EF9F27', label='In cluster'),
          Patch(color='#85B7EB', label='Neighbour')]
ax.legend(handles=legend, frameon=False, fontsize=9)
plt.tight_layout()

## 5. Machine Learning Models

### Why three different models?

| Model | Type | When to use |
|-------|------|-------------|
| **Isolation Forest** | Unsupervised | No labels available; finds statistical outliers |
| **Random Forest** | Supervised | Labels available; interpretable feature importances |
| **Gradient Boosting** | Supervised | Highest precision when labels are clean |

In production, all three are combined into a **weighted ensemble score**.

In [ ]:
from src.models.ml_models import (
    train_isolation_forest, train_random_forest,
    train_gradient_boosting, build_ensemble_scores
)

# Train all models
iso_model, iso_scaler, iso_scores, iso_preds = train_isolation_forest(feature_matrix)
rf_model,  rf_scaler,  rf_results            = train_random_forest(feature_matrix)
gb_model,  gb_scaler,  gb_results            = train_gradient_boosting(feature_matrix)

ensemble = build_ensemble_scores(iso_scores, rf_results['proba'], gb_results['proba'])

In [ ]:
# Feature importance — what drives the RF model's decisions?
fi = rf_results['feat_importance'].sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(9, 6))
fi.plot(kind='barh', ax=ax, color='#185FA5', alpha=0.85, edgecolor='none')
ax.set_title('Top 15 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance score')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()

In [ ]:
# Precision-Recall curves
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(8, 5))
model_results = [
    ('Isolation Forest', iso_scores,          '#888780'),
    ('Random Forest',    rf_results['proba'],  '#185FA5'),
    ('Gradient Boosting',gb_results['proba'], '#1D9E75'),
    ('Ensemble',         ensemble,             '#E24B4A'),
]

fm_idx = feature_matrix.set_index('account')
for name, scores, color in model_results:
    common = scores.index.intersection(fm_idx.index)
    y_true = fm_idx.loc[common, 'is_fraud'].values
    y_prob = scores.loc[common].values
    p, r, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    ax.plot(r, p, label=f'{name} (AP={ap:.3f})', color=color, lw=2)

ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(frameon=False, fontsize=10)
ax.grid(True, alpha=0.2)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()

## 6. Lifecycle Stage Detection

The lifecycle detector classifies every account into one of five stages:

```
Dormant → Recruitment → Activation → Laundering → Exit
```

**Early Detection Gain** measures what fraction of mule accounts we can catch *before* they reach the Laundering stage.

In [ ]:
from src.lifecycle.lifecycle_detector import (
    detect_lifecycle_stages, get_early_stage_accounts, compute_behavioral_drift
)

lifecycle_df = detect_lifecycle_stages(feature_matrix, use_ml=True)
early_df     = get_early_stage_accounts(lifecycle_df)

In [ ]:
# Stage distribution chart
stage_order  = ['Dormant','Recruitment','Activation','Laundering','Exit','Normal']
stage_colors = ['#B4B2A9','#85B7EB','#EF9F27','#E24B4A','#888780','#9FE1CB']

all_counts  = lifecycle_df['lifecycle_stage'].value_counts().reindex(stage_order, fill_value=0)
mule_counts = lifecycle_df[lifecycle_df['is_fraud']==1]['lifecycle_stage'].value_counts().reindex(stage_order, fill_value=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for ax, counts, title in [(ax1, all_counts, 'All accounts'),
                           (ax2, mule_counts,'Confirmed mule accounts')]:
    bars = ax.bar(stage_order, counts.values, color=stage_colors, edgecolor='none', alpha=0.9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=25)
    ax.spines[['top','right']].set_visible(False)
    for bar, v in zip(bars, counts.values):
        if v > 0:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                    str(v), ha='center', va='bottom', fontsize=10)

plt.suptitle('Mule Lifecycle Stage Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()

In [ ]:
# Temporal drift — visualise how a mule account's activity changes over time
mule_accs   = feature_matrix[feature_matrix['is_fraud']==1]['account'].values
normal_accs = feature_matrix[feature_matrix['is_fraud']==0]['account'].values

if len(mule_accs) > 0 and len(normal_accs) > 0:
    mule_drift   = compute_behavioral_drift(df, mule_accs[0])
    normal_drift = compute_behavioral_drift(df, normal_accs[0])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    for ax, drift, title, color in [
        (ax1, mule_drift,   f'Mule: {mule_accs[0][:12]}',   '#E24B4A'),
        (ax2, normal_drift, f'Normal: {normal_accs[0][:12]}','#185FA5'),
    ]:
        if len(drift) > 0:
            ax.bar(range(len(drift)), drift['tx_count'], color=color, alpha=0.75, edgecolor='none')
            ax.plot(range(len(drift)), drift['rolling_mean'], '--', color='#444441', lw=1.5,
                    label='Rolling mean')
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel('Week'); ax.set_ylabel('Transactions sent')
        ax.legend(frameon=False, fontsize=9)
        ax.spines[['top','right']].set_visible(False)

    plt.suptitle('Behavioral Drift: Mule vs Normal Account', fontsize=13, fontweight='bold')
    plt.tight_layout()

## 7. Full Evaluation Summary

In [ ]:
from src.evaluation.evaluator import evaluate_model, evaluate_early_detection
from sklearn.metrics import average_precision_score

fm_indexed = feature_matrix.set_index('account')

results = []
for name, scores in [
    ('IsolationForest',  iso_scores),
    ('RandomForest',     rf_results['proba']),
    ('GradientBoosting', gb_results['proba']),
    ('Ensemble',         ensemble),
]:
    common = scores.index.intersection(fm_indexed.index)
    y_true = fm_indexed.loc[common,'is_fraud'].values
    y_prob = scores.loc[common].values
    y_pred = (y_prob >= 0.5).astype(int)
    r = evaluate_model(y_true, y_pred, y_prob, name, verbose=False)
    results.append(r)

results_df = pd.DataFrame(results)
print('\n=== Model Performance Summary ===')
print(results_df[['Model','Precision','Recall','F1','PR-AUC','ROC-AUC']].to_string(index=False))

print('\n=== Early Detection ===')
edg = evaluate_early_detection(lifecycle_df)

In [ ]:
# Suspicious accounts summary
susp_accs = get_suspicious_accounts(partition, comm_df)
threshold = float(np.percentile(ensemble.values, 95))
ml_susp   = ensemble[ensemble > threshold].index.tolist()
all_susp  = list(set(susp_accs) | set(ml_susp))

print(f'Flagged by community detection : {len(susp_accs):,}')
print(f'Flagged by ML ensemble (top 5%): {len(ml_susp):,}')
print(f'Combined suspicious accounts   : {len(all_susp):,}')
print(f'Early-stage flagged            : {len(early_df):,}')

# Save
pd.Series(all_susp, name='account').to_csv('../outputs/results/notebook_suspicious_accounts.csv', index=False)
print('\nSaved: ../outputs/results/notebook_suspicious_accounts.csv')

## 8. API Scoring Demo

The scoring API can score accounts in real-time once models are trained and loaded. Here we demonstrate the scoring logic directly without starting the HTTP server.

In [ ]:
import pickle
from src.api.scoring_api import ModelRegistry, score_account

# Simulate what the API does at startup: load models from disk
# (This requires you to have run main_pipeline.py first to save models)
registry = ModelRegistry()
# Inject our in-memory objects directly (bypasses file loading for demo)
from src.features.feature_engineering import get_feature_columns
feat_cols = get_feature_columns(feature_matrix)

from sklearn.preprocessing import StandardScaler
scaler_demo = StandardScaler()
scaler_demo.fit(feature_matrix[feat_cols].fillna(0).values)

registry.rf_bundle        = {'model': rf_model,  'scaler': scaler_demo, 'feat_cols': feat_cols}
registry.gb_bundle        = {'model': gb_model,  'scaler': scaler_demo, 'feat_cols': feat_cols}
registry.iso_bundle       = {'model': iso_model, 'scaler': scaler_demo}
registry.feature_matrix   = feature_matrix.set_index('account')
registry.lifecycle_df     = lifecycle_df.set_index('account') if 'account' in lifecycle_df.columns else lifecycle_df
registry._loaded          = True

# Score a known mule account
mule_acc = feature_matrix[feature_matrix['is_fraud']==1]['account'].iloc[0]
result   = score_account(mule_acc, registry)
print('Scoring a confirmed mule account:')
print(f'  Account ID      : {result["account_id"]}')
print(f'  Suspicion score : {result["suspicion_score"]}')
print(f'  Risk tier       : {result["risk_tier"]}')
print(f'  Is flagged      : {result["is_flagged"]}')
print(f'  Lifecycle stage : {result["lifecycle_stage"]}')
print(f'  Top risk factor : {result["top_risk_factors"][0] if result["top_risk_factors"] else "N/A"}')

In [ ]:
# Score a normal account for comparison
normal_acc = feature_matrix[feature_matrix['is_fraud']==0]['account'].iloc[0]
result_n   = score_account(normal_acc, registry)

print('Scoring a normal account:')
print(f'  Account ID      : {result_n["account_id"]}')
print(f'  Suspicion score : {result_n["suspicion_score"]}')
print(f'  Risk tier       : {result_n["risk_tier"]}')
print(f'  Is flagged      : {result_n["is_flagged"]}')
print(f'  Lifecycle stage : {result_n["lifecycle_stage"]}')

In [ ]:
# Score a batch of accounts
batch_accounts = (
    feature_matrix[feature_matrix['is_fraud']==1]['account'].head(5).tolist() +
    feature_matrix[feature_matrix['is_fraud']==0]['account'].head(5).tolist()
)

batch_results = []
for acc in batch_accounts:
    r = score_account(acc, registry, include_explanation=False)
    batch_results.append({
        'account': r['account_id'],
        'score':   r['suspicion_score'],
        'tier':    r['risk_tier'],
        'flagged': r['is_flagged'],
        'stage':   r['lifecycle_stage'],
    })

batch_df = pd.DataFrame(batch_results)
print('Batch scoring results (top 5 = mules, bottom 5 = normal):')
print(batch_df.to_string(index=False))

## Summary

This notebook demonstrated a complete AML mule detection pipeline:

1. **Graph construction** — built a directed transaction graph from raw data
2. **Feature engineering** — extracted 38 behavioral, topological, and temporal features
3. **Community detection** — identified suspicious account clusters using graph clustering
4. **ML models** — trained IsolationForest, RandomForest, GradientBoosting + ensemble
5. **Lifecycle classification** — assigned Dormant/Recruitment/Activation/Laundering/Exit stages
6. **API scoring** — demonstrated real-time account scoring

**Next steps:**
- Run `python main_pipeline.py --source paysim --path data/raw/paysim.csv` for real data
- Enable the GNN module by running `src/gnn/gnn_models.py` (requires torch)
- Start the API server with `python src/api/scoring_api.py`
- Run the full test suite with `python tests/test_pipeline.py`